In [ ]:
# Packages 
import polars as pl

In [ ]:
# Load data
df = pl.scan_csv("../auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [ ]:
# Count number of rows 
#df.select(pl.len()).collect().item()

In [ ]:
# Converting seconds
seconds_per_hour = 60 * 60 
seconds_per_day = 60 * 60 * 24 
seconds_in_ten_days = 60 * 60* 24 * 10

In [42]:
# Day 1 of 58
subset = df.filter(pl.col('time') < seconds_per_day)

In [43]:
# Number of rows in day 1
#subset.select(pl.len()).collect().item()

In [ ]:
# Features dataframe
user_features = (
subset
    .with_columns(
        hour=(pl.col("time") // seconds_per_hour).cast(pl.Int64),
        is_fail=(pl.col("outcome") == "Fail").cast(pl.Int64),
    )
    .group_by(["src_user", "hour"])
    .agg(
        n_auths=pl.len(),
        n_fails=pl.col("is_fail").sum(),
    )
    .with_columns(
        failure_ratio=pl.col("n_fails") / pl.col("n_auths"),
    )
    .collect()
)

In [ ]:
user_features.show()

src_user,hour,n_auths,n_fails,failure_ratio
str,i64,u32,i64,f64
"""C74$@DOM1""",19,42,0,0.0
"""C4499$@DOM1""",6,32,0,0.0
"""U8124@DOM1""",16,58,0,0.0
"""SYSTEM@C9192""",18,2,0,0.0
"""U6604@DOM1""",20,19,0,0.0


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
X = user_features.select(["n_auths", "failure_ratio"]).to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters = 3)
labels = kmeans.fit_predict(X_scaled)

In [ ]:
user_features = user_features.with_columns(cluster = pl.Series(labels))

In [ ]:
user_features.head()

src_user,hour,n_auths,n_fails,failure_ratio,cluster
str,i64,u32,i64,f64,i32
"""C74$@DOM1""",19,42,0,0.0,0
"""C4499$@DOM1""",6,32,0,0.0,0
"""U8124@DOM1""",16,58,0,0.0,0
"""SYSTEM@C9192""",18,2,0,0.0,0
"""U6604@DOM1""",20,19,0,0.0,0


In [ ]:
user_features.group_by("cluster").agg(
    size=pl.len(),
    mean_n_auths=pl.col("n_auths").mean(),
    mean_failure_ratio=pl.col("failure_ratio").mean(),
    max_n_auths=pl.col("n_auths").max(),
).sort("cluster")

cluster,size,mean_n_auths,mean_failure_ratio,max_n_auths
i32,u32,f64,f64,u32
0,317980,42.253453,0.000787,3168
1,352,6294.826705,0.003337,23744
2,6445,13.845772,0.977298,1559


10 DAYS

In [ ]:
# Day 1-10 of 58
subset = df.filter(pl.col('time') < seconds_in_ten_days)

In [ ]:
# Features dataframe
user_features = (
subset
    .with_columns(
        hour=(pl.col("time") // seconds_per_hour).cast(pl.Int64),
        is_fail=(pl.col("outcome") == "Fail").cast(pl.Int64),
    )
    .group_by(["src_user", "hour"])
    .agg(
        n_auths=pl.len(),
        n_fails=pl.col("is_fail").sum(),
    )
    .with_columns(
        failure_ratio=pl.col("n_fails") / pl.col("n_auths"),
    )
    .collect()
)

In [ ]:
X = user_features.select(["n_auths", "failure_ratio"]).to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters = 3)
labels = kmeans.fit_predict(X_scaled)

In [ ]:
user_features = user_features.with_columns(cluster = pl.Series(labels))

In [ ]:
user_features.group_by("cluster").agg(
    size=pl.len(),
    mean_n_auths=pl.col("n_auths").mean(),
    mean_failure_ratio=pl.col("failure_ratio").mean(),
    max_n_auths=pl.col("n_auths").max(),
).sort("cluster")

cluster,size,mean_n_auths,mean_failure_ratio,max_n_auths
i32,u32,f64,f64,u32
0,3815119,39.49787,0.000568,3215
1,2990,6394.748829,0.00473,27381
2,52094,15.368833,0.980093,2786
